In [6]:

import os, csv, json, random
from datetime import date, timedelta

random.seed(42)
os.makedirs("data_tp2", exist_ok=True)

# ---------- produits.csv ----------
catalogue = {
    "Informatique": ["Clavier", "Souris", "Ecran", "Casque", "Webcam"],
    "Maison": ["Lampe", "Chaise", "Bureau", "Etagere"],
    "Sport": ["Ballon", "Raquette", "Tapis de yoga", "Haltères"],
}

produits_l, pid = [], 1

for cat, noms in catalogue.items():
    for n in noms:
        produits_l.append([
            pid, n, cat,
            random.choice([49, 79, 99, 149, 249, 399, 599])
        ])
        pid += 1

produits_l[3][3] = "N/A"  # prix invalide (produit 4)

with open("data_tp2/produits.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id_produit", "nom", "categorie", "prix"])
    w.writerows(produits_l)


# ---------- clients.json (un objet JSON par ligne) ----------
villes = ["Casablanca", "Rabat", "Fès", "Marrakech", "Tanger", "Agadir"]

prenoms = [
    "Sara", "Youssef", "Imane", "Omar", "Salma",
    "Hamza", "Khadija", "Anas", "Meryem", "Ayoub"
]

familles = [
    "Alaoui", "Benani", "Cherkaoui", "Darif",
    "El Idrissi", "Fassi", "Tazi"
]

with open("data_tp2/clients.json", "w", encoding="utf-8") as f:
    for i in range(1, 61):
        ville = random.choice(villes)

        if i % 7 == 0:
            ville = " " + ville.upper() + " "  # ville mal saisie

        c = {
            "id_client": i,
            "nom": random.choice(prenoms) + " " + random.choice(familles),
            "ville": ville,
            "date_inscription": (
                date(2024, 1, 1)
                + timedelta(days=random.randint(0, 364))
            ).isoformat()
        }

        f.write(json.dumps(c, ensure_ascii=False) + "\n")


# ---------- commandes.csv ----------
statuts = [
    "livree", "livree", "livree", "annulee",
    "en_cours", "Livree", "LIVREE "
]

cmd = []

for i in range(1, 3001):
    d = date(2025, 1, 1) + timedelta(days=random.randint(0, 364))

    cmd.append([
        i,
        random.randint(1, 54),
        random.randint(1, 13),
        random.randint(1, 5),
        d.isoformat(),
        random.choice(statuts)
    ])

for k in range(0, 3000, 150):
    cmd[k][3] = "abc"  # quantité invalide

for k in range(5, 3000, 200):
    cmd[k][3] = -2  # quantité négative

for k in range(10, 3000, 300):
    cmd[k][4] = ""  # date manquante

for k in range(20, 3000, 500):
    cmd[k][1] = ""  # client manquant

cmd += random.sample(cmd, 40)  # doublons exacts

with open("data_tp2/commandes.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "id_commande", "id_client", "id_produit",
        "quantite", "date_commande", "statut"
    ])
    w.writerows(cmd)

print(sorted(os.listdir("data_tp2")))


['clients.json', 'commandes.csv', 'produits.csv']


In [8]:
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [11]:
Spark = SparkSession.builder.appName("TP2-Ventes").master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate()
print("Session crée avec succées")


Session crée avec succées


In [13]:
produits_raw = Spark.read.option("header",True).schema("id_produit String ,nom String, categorie String , prix String").csv("data_tp2/produits.csv")
produits_raw.show()
produits_raw.printSchema()

+----------+-------------+------------+----+
|id_produit|          nom|   categorie|prix|
+----------+-------------+------------+----+
|         1|      Clavier|Informatique| 399|
|         2|       Souris|Informatique|  49|
|         3|        Ecran|Informatique|  49|
|         4|       Casque|Informatique| N/A|
|         5|       Webcam|Informatique|  99|
|         6|        Lampe|      Maison|  79|
|         7|       Chaise|      Maison|  79|
|         8|       Bureau|      Maison|  79|
|         9|      Etagere|      Maison| 399|
|        10|       Ballon|       Sport|  49|
|        11|     Raquette|       Sport| 399|
|        12|Tapis de yoga|       Sport| 399|
|        13|     Haltères|       Sport| 249|
+----------+-------------+------------+----+

root
 |-- id_produit: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- prix: string (nullable = true)



In [14]:
clients_raw = Spark.read.json("data_tp2/clients.json")
clients_raw.show()
clients_raw.printSchema()

+----------------+---------+------------------+------------+
|date_inscription|id_client|               nom|       ville|
+----------------+---------+------------------+------------+
|      2024-01-17|        1|       Ayoub Darif|  Casablanca|
|      2024-04-29|        2|    Youssef Benani|  Casablanca|
|      2024-10-14|        3|      Ayoub Alaoui|      Tanger|
|      2024-04-22|        4|      Meryem Darif|       Rabat|
|      2024-01-04|        5|   Ayoub Cherkaoui|   Marrakech|
|      2024-05-22|        6| Khadija Cherkaoui|       Rabat|
|      2024-06-21|        7|         Omar Tazi|      RABAT |
|      2024-02-19|        8|     Youssef Darif|  Casablanca|
|      2024-05-15|        9|  Hamza El Idrissi|         Fès|
|      2024-03-04|       10|   Anas El Idrissi|  Casablanca|
|      2024-05-30|       11|Youssef El Idrissi|   Marrakech|
|      2024-07-04|       12|        Ayoub Tazi|      Agadir|
|      2024-02-05|       13|        Omar Fassi|      Tanger|
|      2024-05-28|      

In [18]:

commandes_raw = Spark.read \
    .option("header", True) \
    .schema("""
        id_commande STRING,
        id_client STRING,
        id_produit STRING,
        quantite STRING,
        date_commande STRING,
        statut STRING
    """) \
    .csv("data_tp2/commandes.csv")

commandes_raw.printSchema()

                                                        

root
 |-- id_commande: string (nullable = true)
 |-- id_client: string (nullable = true)
 |-- id_produit: string (nullable = true)
 |-- quantite: string (nullable = true)
 |-- date_commande: string (nullable = true)
 |-- statut: string (nullable = true)



In [19]:
commandes_raw.show(5)
clients_raw.show(5)
produits_raw.show(5)

+-----------+---------+----------+--------+-------------+-------+
|id_commande|id_client|id_produit|quantite|date_commande| statut|
+-----------+---------+----------+--------+-------------+-------+
|          1|       51|         8|     abc|   2025-05-05| livree|
|          2|        7|        11|       4|   2025-02-18| livree|
|          3|       27|         8|       1|   2025-08-05| Livree|
|          4|       42|         2|       1|   2025-12-01|annulee|
|          5|       52|         2|       2|   2025-06-23| livree|
+-----------+---------+----------+--------+-------------+-------+
only showing top 5 rows
+----------------+---------+---------------+----------+
|date_inscription|id_client|            nom|     ville|
+----------------+---------+---------------+----------+
|      2024-01-17|        1|    Ayoub Darif|Casablanca|
|      2024-04-29|        2| Youssef Benani|Casablanca|
|      2024-10-14|        3|   Ayoub Alaoui|    Tanger|
|      2024-04-22|        4|   Meryem Darif|  

In [20]:

print("Produits :", produits_raw.count())
print("Clients :", clients_raw.count())
print("Commandes :", commandes_raw.count())


Produits : 13
Clients : 60
Commandes : 3040


In [21]:

commandes_raw.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            1
        )
    ).alias(c)
    for c in commandes_raw.columns
]).show()


+-----------+---------+----------+--------+-------------+------+
|id_commande|id_client|id_produit|quantite|date_commande|statut|
+-----------+---------+----------+--------+-------------+------+
|          0|        7|         0|       0|           10|     0|
+-----------+---------+----------+--------+-------------+------+



In [25]:
total_commandes = commandes_raw.count()
doublons = commandes_raw.distinct().count()
nbr_cd = total_commandes - doublons
print(nbr_cd)

40


In [26]:

commandes_raw.groupBy("statut").count().show()


+--------+-----+
|  statut|count|
+--------+-----+
|en_cours|  413|
|  Livree|  465|
| annulee|  476|
| LIVREE |  426|
|  livree| 1260|
+--------+-----+



In [27]:

quantites_invalides = commandes_raw.filter(
    F.col("quantite").try_cast("int").isNull() |
    (F.col("quantite").try_cast("int") < 0)
)

print("Nombre de lignes invalides :", quantites_invalides.count())

quantites_invalides.show()


Nombre de lignes invalides : 35
+-----------+---------+----------+--------+-------------+--------+
|id_commande|id_client|id_produit|quantite|date_commande|  statut|
+-----------+---------+----------+--------+-------------+--------+
|          1|       51|         8|     abc|   2025-05-05|  livree|
|          6|       35|         8|      -2|   2025-04-08| annulee|
|        151|       25|         8|     abc|   2025-09-25|  livree|
|        206|       44|         5|      -2|   2025-11-20|  Livree|
|        301|       43|         5|     abc|   2025-09-27|  livree|
|        406|       41|         7|      -2|   2025-12-10|en_cours|
|        451|        4|         5|     abc|   2025-02-19| LIVREE |
|        601|       18|        13|     abc|   2025-07-01|  livree|
|        606|        1|        12|      -2|   2025-11-23|  livree|
|        751|       32|         7|     abc|   2025-05-03|  livree|
|        806|        6|         3|      -2|   2025-03-03| LIVREE |
|        901|        7|       

In [36]:
prix_invalides = produits_raw.filter(F.col("prix").try_cast("double").isNull())
prix_invalides.show()
prix_invalides.count()

                                    

+----------+------+------------+----+
|id_produit|   nom|   categorie|prix|
+----------+------+------------+----+
|         4|Casque|Informatique| N/A|
+----------+------+------------+----+



1